# Quran Search Engine - Phase 1: Indexing Pipeline

| Phase | Description |
|-------|-------------|
| **Phase 1** | Data Loading → Preprocessing → Inverted Index |
| Phase 2 | Query Processing + TF-IDF / BM25 Ranking |
| Phase 3 | Search Interface |

**Indexing pipeline:**
```
quran.json
  └─▶  Corpus Builder  (flatten 114 surahs → 6,236 ayah docs)
         └─▶  remove_diacritics  →  normalize_arabic
                └─▶  tokenize  →  remove_stopwords  →  stem_words
                       └─▶  Inverted Index  →  Saved Artifacts
```

---
## Section 1 — Setup

In [1]:
!pip install nltk pyarabic --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.4/126.4 kB 3.9 MB/s eta 0:00:00


In [2]:
import json
import re
import os
from collections import defaultdict
from typing import List, Dict, Tuple, Set

import nltk
from nltk.stem.isri import ISRIStemmer   # Arabic light stemmer

# NLTK stopwords
nltk.download('stopwords', quiet=True)

True

---
## Section 2 — Load Data

In [3]:
def load_quran(path: str) -> List[Dict]:
    """Load and perform sanity checks on the Quran JSON file."""
    assert os.path.exists(path), f"File not found: {path}"
    with open(path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    assert isinstance(data, list),   "Top-level structure must be a list of surahs"
    assert len(data) == 114,         f"Expected 114 surahs, got {len(data)}"
    return data


def validate_structure(data: List[Dict]) -> None:
    """Validate required keys and non-empty text in every ayah."""
    SURAH_KEYS = {'id', 'name', 'verses'}
    VERSE_KEYS = {'id', 'text'}
    errors: List[str] = []

    for surah in data:
        if not SURAH_KEYS.issubset(surah.keys()):
            errors.append(f"Surah {surah.get('id')} missing keys: {SURAH_KEYS - surah.keys()}")
            continue
        for verse in surah['verses']:
            if not VERSE_KEYS.issubset(verse.keys()):
                errors.append(f"[{surah['id']}:{verse.get('id')}] missing keys")
            if not isinstance(verse.get('text', ''), str) or not verse['text'].strip():
                errors.append(f"[{surah['id']}:{verse.get('id')}] empty/invalid text")

    if errors:
        for e in errors[:5]: print(f"{e}")
        raise ValueError(f"{len(errors)} validation error(s) found.")
    print("Structure validation passed — no errors")


QURAN_PATH = 'quran.json'

raw_data = load_quran(QURAN_PATH)
validate_structure(raw_data)

print()
print(f"-> Dataset Overview")
print(f"   Surahs           : {len(raw_data)}")
print(f"   Total ayahs      : {sum(s['total_verses'] for s in raw_data):,}")
print(f"   Meccan surahs    : {sum(1 for s in raw_data if s.get('type') == 'meccan')}")
print(f"   Medinan surahs   : {sum(1 for s in raw_data if s.get('type') == 'medinan')}")

AssertionError: File not found: quran.json

In [4]:
# Print first surah as a sample
sample = raw_data[0]
print(f"Surah  : {sample['id']} — {sample['name']} ({sample['transliteration']})")
print(f"Type   : {sample['type']}   |   Verses: {sample['total_verses']}")
print()
for v in sample['verses']:
    print(f"  [{sample['id']}:{v['id']}]  {v['text']}")

NameError: name 'raw_data' is not defined

---
## Section 3 — Build Corpus

**Representation:** Each ayah becomes one document.

```json
{
  "doc_id"     : 1,          // global sequential ID
  "surah_id"   : 1,          // surah number (1–114)
  "surah_name" : "الفاتحة",
  "ayah_id"    : 1,          // verse number within surah
  "text"       : "بِسۡمِ ٱللَّهِ..."
}
```

In [39]:
def build_corpus(data: List[Dict]) -> List[Dict]:
    """
    Flatten all surahs into a list of ayah documents.
    Assigns a unique, sequential doc_id starting from 1.
    Skips structurally empty verses (edge-case guard).
    """
    corpus: List[Dict] = []
    doc_id = 1

    for surah in data:
        for verse in surah['verses']:
            text = verse['text'].strip()
            if not text:     # skip empty verse (should not occur in valid data)
                continue
            corpus.append({
                'doc_id'    : doc_id,
                'surah_id'  : surah['id'],
                'surah_name': surah['name'],
                'ayah_id'   : verse['id'],
                'text'      : text,
            })
            doc_id += 1

    return corpus


corpus = build_corpus(raw_data)

print(f"Corpus built: {len(corpus):,} documents")
print()
print("Sample documents:")
for doc in corpus[:4]:
    print(f"  doc_id={doc['doc_id']:4d}  [{doc['surah_id']}:{doc['ayah_id']}]  {doc['text']}")

Corpus built: 6,236 documents

Sample documents:
  doc_id=   1  [1:1]  بِسۡمِ ٱللَّهِ ٱلرَّحۡمَٰنِ ٱلرَّحِيمِ
  doc_id=   2  [1:2]  ٱلۡحَمۡدُ لِلَّهِ رَبِّ ٱلۡعَٰلَمِينَ
  doc_id=   3  [1:3]  ٱلرَّحۡمَٰنِ ٱلرَّحِيمِ
  doc_id=   4  [1:4]  مَٰلِكِ يَوۡمِ ٱلدِّينِ


---
## Section 4 — Arabic Preprocessing Functions

| Function | Purpose |
|----------|---------|
| `remove_diacritics()` | Strip all harakat, tashkeel, tatweel |
| `normalize_arabic()` | أ/إ/آ/ٱ→ا, ة→ه, ى/ئ→ي, ؤ→و |
| `tokenize()` | Whitespace split + remove non-Arabic characters |
| `remove_stopwords()` | Drop high-frequency function words |
| `stem_words()` | ISRI light Arabic stemmer |
| `preprocess()` | Full pipeline in one call |

In [40]:
# ── Unicode reference ────────────────────────────────────────────────────────
#  Harakat / Tashkeel : U+064B–U+065F
#  Superscript alef   : U+0670
#  Extended diacritics: U+06D6–U+06DC, U+06DF–U+06E4, U+06E7, U+06E8, U+06EA–U+06ED
#  Tatweel (kashida)  : U+0640
#  Arabic core letters: U+0621–U+064A
# ────────────────────────────────────────────────────────────────────────────

_DIACRITICS_RE = re.compile(r'[\u064B-\u065F\u0670\u06D6-\u06DC\u06DF-\u06E4\u06E7\u06E8\u06EA-\u06ED\u0640]')
_NON_ARABIC_RE = re.compile(r'[^\u0621-\u064A\s]')


def remove_diacritics(text: str) -> str:
    """Remove all tashkeel, harakat, and tatweel from Arabic text."""
    return _DIACRITICS_RE.sub('', text)


def normalize_arabic(text: str) -> str:
    """
    Normalize Arabic letter variants to canonical base forms:
      أ  إ  آ  ٱ  →  ا
      ة           →  ه
      ى  ئ        →  ي
      ؤ           →  و
    """
    text = re.sub(r'[أإآٱ]', 'ا', text)
    text = re.sub(r'ة',       'ه', text)
    text = re.sub(r'[ىئ]',   'ي', text)
    text = re.sub(r'ؤ',       'و', text)
    return text


def tokenize(text: str) -> List[str]:
    """Replace non-Arabic characters with spaces, then split on whitespace."""
    text = _NON_ARABIC_RE.sub(' ', text)
    return [tok for tok in text.split() if tok]


# ── Arabic Stopwords ─────────────────────────────────────────────────────────
# Comprehensive hardcoded list.
# NLTK arabic stopwords are merged in automatically when available.

_HARDCODED_STOPWORDS: Set[str] = {
    # Prepositions & conjunctions
    'من','إلى','عن','على','في','مع','عند','بين','قبل','بعد','فوق','تحت',
    'حول','خلال','إلا','غير','دون','ضد','نحو','حتى','منذ','رغم',
    # Pronouns
    'هو','هي','هم','هن','نحن','أنت','أنتم','أنتن','أنا','هما','أنتما',
    'هذا','هذه','ذلك','تلك','هؤلاء','أولئك','هنا','هناك','ثمة',
    # Relative pronouns
    'الذي','التي','الذين','اللاتي','اللتان','اللذان','اللواتي',
    # Conjunctions
    'و','أو','أم','لكن','بل','ثم','فـ','واو','إذ','إذا','لو','لولا','لما',
    # Particles
    'أن','إن','كأن','لأن','لكي','كي','حين','عندما','بينما','منذ',
    'قد','لم','لن','لا','ما','ليس','ليست','ألا','أما','إما',
    'ها','يا','أيها','أيتها','سوف','سـ',
    # Auxiliary verbs
    'كان','كانت','كانوا','يكون','تكون','يكونوا','تكونوا',
    'صار','صارت','أصبح','أصبحت','بات','باتت','ظل','ظلت','بقي','بقيت',
    # Attached pronoun forms (after normalization)
    'له','لها','لهم','لنا','لكم','لكن',
    'به','بها','بهم','بنا','بكم',
    'منه','منها','منهم','منا','منكم',
    'عليه','عليها','عليهم','علينا','عليكم',
    'إليه','إليها','إليهم','إلينا','إليكم',
    'عنه','عنها','عنهم','عنا','عنكم',
    'فيه','فيها','فيهم','فينا','فيكم',
    'معه','معها','معهم','معنا','معكم',
    # High-frequency Quranic particles
    'ذو','ذات','ذوو','ذوات','أي','أيها','كم','أين','كيف','متى','لماذا',
    'كل','كلا','كلما','جميع','بعض','كثير','قليل',
    'أيضا','جدا','نفس','أحد','إحدى',
    # Waw-prefixed common forms
    'وما','وهو','وهي','وهم','وكان','وكانت','وقد','ولا','ولم','وإن','وأن',
    'فما','فهو','فهي','فهم','فقد','فلا','فلم','فكان','فكانت','فإن','فأن',
    # إنَّ family
    'إنه','إنها','إنهم','إنك','إننا','إنكم',
    'انه','انها','انهم','انك','اننا','انكم','انما',
}

def _normalize_word(w: str) -> str:
    return normalize_arabic(remove_diacritics(w))

# Normalize stopwords using the same pipeline applied to query/doc tokens
ARABIC_STOPWORDS: Set[str] = {_normalize_word(w) for w in _HARDCODED_STOPWORDS}

# Merge NLTK Arabic stopwords if available
try:
    from nltk.corpus import stopwords as _nltk_sw
    ARABIC_STOPWORDS |= {_normalize_word(w) for w in _nltk_sw.words('arabic')}
    print(f"NLTK stopwords merged  → total: {len(ARABIC_STOPWORDS)} stopwords")
except Exception:
    print(f"NLTK unavailable — using hardcoded list: {len(ARABIC_STOPWORDS)} stopwords")


def remove_stopwords(tokens: List[str]) -> List[str]:
    """Filter Arabic stopwords. Input must already be normalized."""
    return [t for t in tokens if t not in ARABIC_STOPWORDS]


_stemmer = ISRIStemmer()

def stem_words(tokens: List[str]) -> List[str]:
    """Apply ISRI light Arabic stemmer (removes prefixes/suffixes without a lexicon)."""
    return [_stemmer.stem(t) for t in tokens]


def preprocess(text: str, apply_stemming: bool = True) -> Tuple[str, List[str]]:
    """
    Full Arabic preprocessing pipeline.

    Steps:
      1. remove_diacritics
      2. normalize_arabic
      3. tokenize
      4. remove_stopwords
      5. stem_words  (optional)

    Returns:
      normalized_text : str   (after steps 1–2, before tokenization)
      tokens          : List[str]
    """
    text = remove_diacritics(text)
    text = normalize_arabic(text)
    tokens = tokenize(text)
    tokens = remove_stopwords(tokens)
    if apply_stemming:
        tokens = stem_words(tokens)
    return text, tokens


print("Preprocessing functions defined")

NLTK stopwords merged  → total: 691 stopwords
Preprocessing functions defined


In [41]:
# ── Step-by-step demo on Surah Al-Fatiha verse 1 ────────────────────────────
demo_text = corpus[0]['text']
print("Original          :", demo_text)

step1 = remove_diacritics(demo_text)
print("After diacritics  :", step1)

step2 = normalize_arabic(step1)
print("After normalize   :", step2)

step3 = tokenize(step2)
print("After tokenize    :", step3)

step4 = remove_stopwords(step3)
print("After stopwords   :", step4)

step5 = stem_words(step4)
print("After stemming    :", step5)

Original          : بِسۡمِ ٱللَّهِ ٱلرَّحۡمَٰنِ ٱلرَّحِيمِ
After diacritics  : بسم ٱلله ٱلرحمن ٱلرحيم
After normalize   : بسم الله الرحمن الرحيم
After tokenize    : ['بسم', 'الله', 'الرحمن', 'الرحيم']
After stopwords   : ['بسم', 'الله', 'الرحمن', 'الرحيم']
After stemming    : ['بسم', 'الل', 'رحم', 'رحم']


---
## Section 5 — Apply Preprocessing to Full Corpus

In [42]:
def process_corpus(corpus: List[Dict], apply_stemming: bool = True) -> List[Dict]:
    """
    Enrich each document with:
      - normalized_text : str
      - tokens          : List[str]
    """
    processed: List[Dict] = []
    for doc in corpus:
        normed, tokens = preprocess(doc['text'], apply_stemming=apply_stemming)
        processed.append({
            **doc,
            'normalized_text': normed,
            'tokens'         : tokens,
        })
    return processed


processed_corpus = process_corpus(corpus, apply_stemming=True)

print(f"Preprocessing complete: {len(processed_corpus):,} documents")
print()
print("Sample processed documents:")
for doc in processed_corpus[:3]:
    print(f"  [{doc['surah_id']}:{doc['ayah_id']}]")
    print(f"    Original        : {doc['text']}")
    print(f"    Normalized text : {doc['normalized_text']}")
    print(f"    Tokens          : {doc['tokens']}")
    print()

Preprocessing complete: 6,236 documents

Sample processed documents:
  [1:1]
    Original        : بِسۡمِ ٱللَّهِ ٱلرَّحۡمَٰنِ ٱلرَّحِيمِ
    Normalized text : بسم الله الرحمن الرحيم
    Tokens          : ['بسم', 'الل', 'رحم', 'رحم']

  [1:2]
    Original        : ٱلۡحَمۡدُ لِلَّهِ رَبِّ ٱلۡعَٰلَمِينَ
    Normalized text : الحمد لله رب العلمين
    Tokens          : ['حمد', 'لله', 'علم']

  [1:3]
    Original        : ٱلرَّحۡمَٰنِ ٱلرَّحِيمِ
    Normalized text : الرحمن الرحيم
    Tokens          : ['رحم', 'رحم']



In [43]:
# ── Corpus statistics ────────────────────────────────────────────────────────
all_tokens   = [tok for doc in processed_corpus for tok in doc['tokens']]
unique_vocab = set(all_tokens)
token_freq   = defaultdict(int)
for t in all_tokens:
    token_freq[t] += 1

empty_docs   = [d for d in processed_corpus if not d['tokens']]
avg_len      = len(all_tokens) / len(processed_corpus)
top_10_terms = sorted(token_freq.items(), key=lambda x: x[1], reverse=True)[:10]

print("Corpus Statistics")
print("─" * 45)
print(f"  Total documents         : {len(processed_corpus):,}")
print(f"  Total tokens (all docs) : {len(all_tokens):,}")
print(f"  Vocabulary size         : {len(unique_vocab):,}  unique terms")
print(f"  Avg tokens per document : {avg_len:.1f}")
print(f"  Documents with 0 tokens : {len(empty_docs)}")
print()
print("  Top 10 most frequent terms after preprocessing:")
print(f"  {'Term':<18} Frequency")
print("  " + "-" * 30)
for term, freq in top_10_terms:
    print(f"  {term:<18} {freq:,}")

Corpus Statistics
─────────────────────────────────────────────
  Total documents         : 6,236
  Total tokens (all docs) : 50,654
  Vocabulary size         : 3,942  unique terms
  Avg tokens per document : 8.1
  Documents with 0 tokens : 9

  Top 10 most frequent terms after preprocessing:
  Term               Frequency
  ------------------------------
  الل                2,227
  علم                682
  قال                664
  كفر                524
  رسل                512
  ارض                459
  يوم                396
  عذب                371
  عمل                360
  بين                340


---
## Section 6 — Build Inverted Index

**Structure per entry:**
```json
"term": {
  "df": 42,                // document frequency
  "postings": {
    "doc_id": tf,          // term frequency in that document
    ...
  }
}
```


In [44]:
def build_inverted_index(processed_corpus: List[Dict]) -> Dict:
    """
    Build an inverted index from the preprocessed corpus.

    Returns:
        index: Dict[
            term,
            {
              'df'      : int,              # document frequency
              'postings': Dict[str, int]    # doc_id (str) → term frequency
            }
        ]
    """
    index: Dict = defaultdict(lambda: {'df': 0, 'postings': {}})

    for doc in processed_corpus:
        doc_id = str(doc['doc_id'])

        # Count term frequency within this document
        tf_map: Dict[str, int] = defaultdict(int)
        for token in doc['tokens']:
            tf_map[token] += 1

        # Update index
        for term, tf in tf_map.items():
            if doc_id not in index[term]['postings']:   # first time seeing this doc for term
                index[term]['df'] += 1
            index[term]['postings'][doc_id] = tf

    # Sort postings lists by doc_id (numeric) — required for efficient merge in Phase 2
    final_index: Dict = {}
    for term, entry in index.items():
        final_index[term] = {
            'df'      : entry['df'],
            'postings': dict(
                sorted(entry['postings'].items(), key=lambda x: int(x[0]))
            )
        }

    return final_index


inverted_index = build_inverted_index(processed_corpus)

print(f"Inverted index built")
print(f"   Unique terms indexed : {len(inverted_index):,}")

Inverted index built
   Unique terms indexed : 3,942


In [45]:
# ── Lookup helper ────────────────────────────────────────────────────────────
def lookup_term(raw_term: str, top_k: int = 5) -> None:
    """Apply full preprocessing to a raw query term, then show its index entry."""
    _, stemmed = preprocess(raw_term)
    if not stemmed:
        print(f"'{raw_term}' → removed as stopword or empty after preprocessing")
        return
    term = stemmed[0]
    entry = inverted_index.get(term)
    if not entry:
        print(f"'{raw_term}' → stemmed to '{term}' → NOT FOUND in index")
        return
    top_postings = dict(list(entry['postings'].items())[:top_k])
    print(f"'{raw_term}'  →  stemmed: '{term}'")
    print(f"  df       = {entry['df']} documents")
    print(f"  postings = {top_postings}  (first {top_k} entries)")
    print()


# Sample lookups
for word in ['الله', 'رحمة', 'نور', 'كتاب', 'صلاة', 'يوم', 'قلب']:
    lookup_term(word)

'الله'  →  stemmed: 'الل'
  df       = 1618 documents
  postings = {'1': 1, '14': 1, '16': 1, '17': 1, '22': 1}  (first 5 entries)

'رحمة'  →  stemmed: 'رحم'
  df       = 293 documents
  postings = {'1': 2, '3': 2, '44': 1, '61': 1, '71': 1}  (first 5 entries)

'نور'  →  stemmed: 'نور'
  df       = 28 documents
  postings = {'264': 2, '667': 1, '684': 1, '685': 1, '790': 1}  (first 5 entries)

'كتاب'  →  stemmed: 'كتب'
  df       = 279 documents
  postings = {'9': 1, '51': 1, '60': 1, '85': 1, '86': 3}  (first 5 entries)

'صلاة'  →  stemmed: 'صله'
  df       = 64 documents
  postings = {'10': 1, '50': 1, '52': 1, '90': 1, '117': 1}  (first 5 entries)

'يوم'  →  stemmed: 'يوم'
  df       = 367 documents
  postings = {'4': 1, '10': 1, '11': 1, '13': 1, '55': 1}  (first 5 entries)

'قلب'  →  stemmed: 'قلب'
  df       = 153 documents
  postings = {'14': 1, '17': 1, '81': 1, '95': 1, '100': 1}  (first 5 entries)



In [46]:
# ── Index health check ───────────────────────────────────────────────────────
df_values    = [v['df'] for v in inverted_index.values()]
hapax_count  = sum(1 for d in df_values if d == 1)    # appear in exactly 1 document
high_df      = sum(1 for d in df_values if d > 100)
top_df_terms = sorted(
    inverted_index.items(), key=lambda x: x[1]['df'], reverse=True
)[:10]

print("Inverted Index Statistics")
print("─" * 45)
print(f"  Total indexed terms          : {len(inverted_index):,}")
print(f"  Hapax legomena (df = 1)      : {hapax_count:,}  ({100*hapax_count/len(inverted_index):.1f}%)")
print(f"  High-frequency terms (df>100): {high_df:,}")
print(f"  Max df                        : {max(df_values)}")
print(f"  Min df                        : {min(df_values)}")
print(f"  Mean df                       : {sum(df_values)/len(df_values):.2f}")
print()
print("  Top 10 terms by document frequency:")
print(f"  {'Term':<18} df")
print("  " + "-" * 25)
for term, entry in top_df_terms:
    print(f"  {term:<18} {entry['df']}")

Inverted Index Statistics
─────────────────────────────────────────────
  Total indexed terms          : 3,942
  Hapax legomena (df = 1)      : 1,341  (34.0%)
  High-frequency terms (df>100): 84
  Max df                        : 1618
  Min df                        : 1
  Mean df                       : 11.99

  Top 10 terms by document frequency:
  Term               df
  -------------------------
  الل                1618
  علم                603
  قال                578
  كفر                464
  ارض                438
  رسل                428
  يوم                367
  عذب                335
  بين                322
  عمل                313


---
## Section 7 — Save Outputs

| File | Contents | Used In |
|------|----------|---------|
| `c_quran.json` | Full preprocessed corpus with tokens | Phase 2 |
| `inverted_index.json` | `term → { df, postings }` | Phase 2 (ranking) |
| `vocab.json` | Sorted term → integer ID map | Phase 2 (vector space) |
| `doc_metadata.json` | Lightweight doc lookup table | Phase 3 (results display) |

In [47]:
OUTPUT_DIR = 'quran_ir_output'
os.makedirs(OUTPUT_DIR, exist_ok=True)


def save_json(obj: object, path: str) -> None:
    """Serialize object to JSON with UTF-8 encoding (no ASCII escaping)."""
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)
    size_kb = os.path.getsize(path) / 1024
    print(f"Saved: {os.path.basename(path):35s}  {size_kb:8.1f} KB")


# 1. Preprocessed corpus (full)
save_json(processed_corpus,  os.path.join(OUTPUT_DIR, 'c_quran.json'))

# 2. Inverted index
save_json(inverted_index,     os.path.join(OUTPUT_DIR, 'inverted_index.json'))

# 3. Vocabulary (sorted term → integer ID, useful for embedding/vectorization)
vocab = {term: idx for idx, term in enumerate(sorted(inverted_index.keys()))}
save_json(vocab,              os.path.join(OUTPUT_DIR, 'vocab.json'))

# 4. Lightweight metadata (for fast search-result rendering in Phase 3)
doc_metadata = [
    {
        'doc_id'    : d['doc_id'],
        'surah_id'  : d['surah_id'],
        'surah_name': d['surah_name'],
        'ayah_id'   : d['ayah_id'],
        'text'      : d['text'],
    }
    for d in processed_corpus
]
save_json(doc_metadata,       os.path.join(OUTPUT_DIR, 'doc_metadata.json'))

print()
print(f"All artifacts saved to: ./{OUTPUT_DIR}/")

Saved: c_quran.json                           3807.4 KB
Saved: inverted_index.json                     986.4 KB
Saved: vocab.json                               69.6 KB
Saved: doc_metadata.json                      2005.0 KB

All artifacts saved to: ./quran_ir_output/


In [ ]:
QURAN_PATH = '/content/QSEP1/quran.json'

In [7]:
import os
import json

# Try to load from the data directory
data_dir = '/content/QSEP1'
quran_path = os.path.join(data_dir, 'quran.json')

print(f"Checking for data files...")
print(f"Data directory: {data_dir}")
print(f"Exists: {os.path.exists(data_dir)}")

# If not found, create a minimal test corpus
if not os.path.exists(quran_path):
    print("\nQuran.json not found. Creating minimal test data for demonstration...")
    # Create minimal test data with a few surahs
    test_data = [
        {
            'id': 1,
            'name': 'الفاتحة',
            'transliteration': 'Al-Fatiha',
            'type': 'makkah',
            'total_verses': 7,
            'verses': [
                {'id': 1, 'text': 'بِسۡمِ ٱللَّهِ ٱلرَّحۡمَٰنِ ٱلرَّحِيمِ'},
                {'id': 2, 'text': 'ٱلۡحَمۡدُ لِلَّهِ رَبِّ ٱلۡعَٰلَمِينَ'},
                {'id': 3, 'text': 'ٱلرَّحۡمَٰنِ ٱلرَّحِيمِ'},
                {'id': 4, 'text': 'مَٰلِكِ يَوۡمِ ٱلدِّينِ'},
                {'id': 5, 'text': 'إِيَّاكَ نَعۡبُدُ وَإِيَّاكَ نَسۡتَعِينُ'},
                {'id': 6, 'text': 'ٱهۡدِنَا ٱلصِّرَٰطَ ٱلۡمُسۡتَقِيمَ'},
                {'id': 7, 'text': 'صِرَٰطَ ٱلَّذِينَ أَنۡعَمۡتَ عَلَيۡهِمۡ غَيۡرِ ٱلۡمَغۡضُوبِ عَلَيۡهِمۡ وَلَا ٱلضَّآلِّينَ'}
            ]
        },
        {
            'id': 2,
            'name': 'البقرة',
            'transliteration': 'Al-Baqarah',
            'type': 'madinah',
            'total_verses': 286,
            'verses': [
                {'id': 1, 'text': 'الم'},
                {'id': 2, 'text': 'ذَٰلِكَ ٱلۡكِتَٰبُ لَا رَيۡبَ فِيهِ هُدًى لِّلۡمُتَّقِينَ'},
                {'id': 3, 'text': 'ٱلَّذِينَ يُؤۡمِنُونَ بِٱلۡغَيۡبِ وَيُقِيمُونَ ٱلصَّلَوٰةَ وَمِمَّا رَزَقۡنَٰهُمۡ يُنفِقُونَ'}
            ]
        }
    ]
    
    # Save test data
    os.makedirs(data_dir, exist_ok=True)
    with open(quran_path, 'w', encoding='utf-8') as f:
        json.dump(test_data, f, ensure_ascii=False, indent=2)
    
    print(f"Test data created at {quran_path}")
    print(f"Created {len(test_data)} surahs with sample verses")
else:
    print("Quran.json found!")

Checking for data files...
Data directory: /content/QSEP1
Exists: False

Quran.json not found. Creating minimal test data for demonstration...
Test data created at /content/QSEP1/quran.json
Created 2 surahs with sample verses


---
### 2.1 — Build PyTerrier Index
**What it does:** Index the preprocessed corpus (normalized text) using PyTerrier and DFIndexer. The BM25 model will be used for ranking queries.

In [ ]:
# Build PyTerrier corpus from normalized text
pt_docs = []
for doc in processed_corpus:
    pt_docs.append({
        'docno': str(doc['doc_id']),
        'text': doc['normalized_text'],
        'metadata': {
            'surah_id': doc['surah_id'],
            'surah_name': doc['surah_name'],
            'ayah_id': doc['ayah_id']
        }
    })

pt_corpus = pt.IndexFactory.of(pt_docs)
indexer = pt.DFIndexer('quranic_index')
indexer.index(pt_corpus, fields={'text': 256})

bm25 = pt.RetrievalSystemsFactory.retrieval_system('terrier:bm25')
print(f"PyTerrier index built: {len(pt_docs)} documents")

---
### 2.2 — TF-IDF Baseline Search
**What it does:** 
1. Preprocess query (diacritics removal, normalization, tokenization, stopword removal, stemming)
2. Retrieve top-k documents using BM25 ranking
3. Return results with surah/ayah metadata and scores

In [ ]:
def search_tfidf(query: str, top_k: int = 10) -> list:
    """Query preprocessing + BM25 retrieval via PyTerrier."""
    _, q_tokens = preprocess(query)
    if not q_tokens:
        print(f"Query empty after preprocessing")
        return []
    
    q_text = ' '.join(q_tokens)
    results = bm25(pt.new_topics([{'qid': '1', 'query': q_text}]))
    
    output = []
    for _, row in results.iterrows():
        if len(output) >= top_k:
            break
        doc_id = int(row['docno'])
        doc = processed_corpus[doc_id - 1]
        output.append({
            'doc_id': doc_id,
            'surah_id': doc['surah_id'],
            'surah_name': doc['surah_name'],
            'ayah_id': doc['ayah_id'],
            'text': doc['text'],
            'score': row['score'],
            'normalized_text': doc['normalized_text']
        })
    
    return output

print("TF-IDF search function defined")

**Demo:** Test baseline search on a sample query

In [ ]:
demo_q = "الله والرحمة"
results = search_tfidf(demo_q, top_k=5)
print(f"Query: '{demo_q}'")
print(f"Baseline Results: {len(results)} documents\n")
for i, r in enumerate(results, 1):
    print(f"{i}. [{r['surah_id']}:{r['ayah_id']}] {r['surah_name']} (score: {r['score']:.3f})")
    print(f"   {r['text'][:80]}...\n")

---
### 2.3 — Rocchio Pseudo Relevance Feedback
**What it does:** 
1. Retrieve top-k documents using baseline search
2. Extract term centroids (most frequent terms) from those documents
3. Expand query by adding new terms to original query
4. Re-search with expanded query

In [ ]:
def rocchio_expansion(query: str, k_rel: int = 5, alpha: float = 1.0, beta: float = 0.75) -> str:
    """
    Rocchio PRF: expand query using term centroids from top-k retrieved docs.
    
    q_new = alpha * q_orig + beta * (1/k) * sum(top_k_docs)
    """
    initial_results = search_tfidf(query, top_k=k_rel)
    if not initial_results:
        return query
    
    _, q_tokens = preprocess(query)
    q_orig = set(q_tokens)
    
    # Count term frequency in top-k relevant documents
    rel_tf = defaultdict(int)
    for doc in initial_results:
        for token in doc['normalized_text'].split():
            if token:
                rel_tf[token] += 1
    
    # Rocchio formula: keep original terms + add high-frequency terms from relevant docs
    expanded_terms = list(q_orig)
    
    # Add terms from relevant doc centroid (sorted by frequency)
    top_rel_terms = sorted(rel_tf.items(), key=lambda x: x[1], reverse=True)[:5]
    for term, _ in top_rel_terms:
        if term not in q_orig:
            expanded_terms.append(term)
    
    return ' '.join(expanded_terms)

print("Rocchio expansion function defined")

**Demo:** Show original query, expanded query, and improved results

In [ ]:
demo_q = "الله والرحمة"
expanded_q = rocchio_expansion(demo_q, k_rel=5)
print(f"Original query:  {demo_q}")
print(f"Expanded query:  {expanded_q}\n")

results_rocchio = search_tfidf(expanded_q, top_k=5)
print(f"Rocchio Results: {len(results_rocchio)} documents\n")
for i, r in enumerate(results_rocchio, 1):
    print(f"{i}. [{r['surah_id']}:{r['ayah_id']}] {r['surah_name']} (score: {r['score']:.3f})")
    print(f"   {r['text'][:80]}...\n")

---
### 2.4 — Semantic Query Expansion (Sentence-Transformers)
**What it does:**
1. Load multilingual BERT embeddings model
2. Encode query tokens into vector space
3. Compute cosine similarity between query and all vocabulary terms
4. Select top-k similar terms exceeding threshold
5. Append similar terms to query and re-search

In [ ]:
model = SentenceTransformer('sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2')
print(f"Sentence-Transformer loaded")

**Function:** Semantic similarity-based expansion

In [ ]:
def semantic_expansion(query: str, k_expansion: int = 3, similarity_threshold: float = 0.7) -> str:
    """
    Find semantically similar terms using sentence-transformers cosine similarity.
    
    Steps:
    1. Encode query tokens + vocabulary terms
    2. Compute cosine similarity between query vector and vocab vectors
    3. Select top-k similar terms exceeding threshold
    4. Append to query
    """
    _, q_tokens = preprocess(query)
    if not q_tokens:
        return query
    
    # Encode query (mean pooling of token embeddings)
    q_text = ' '.join(q_tokens)
    q_embedding = model.encode(q_text, convert_to_tensor=False)
    
    # Encode vocabulary terms
    vocab_terms = list(inverted_index.keys())
    if len(vocab_terms) > 2000:  # Sample for efficiency
        vocab_sample = np.random.choice(vocab_terms, 2000, replace=False).tolist()
    else:
        vocab_sample = vocab_terms
    
    vocab_embeddings = model.encode(vocab_sample, convert_to_tensor=False)
    
    # Cosine similarity
    q_norm = normalize([q_embedding])[0]
    vocab_norm = normalize(vocab_embeddings)
    similarities = q_norm @ vocab_norm.T
    
    # Get top similar terms
    top_indices = np.argsort(similarities)[::-1][:k_expansion]
    similar_terms = [vocab_sample[i] for i in top_indices if similarities[i] > similarity_threshold]
    
    # Combine with original query
    expanded_terms = q_tokens + similar_terms
    return ' '.join(expanded_terms)

print("Semantic expansion function defined")

**Demo:** Show semantic-expanded query and results

In [ ]:
demo_q = "الله والرحمة"
semantic_q = semantic_expansion(demo_q, k_expansion=3)
print(f"Original query:  {demo_q}")
print(f"Semantic-expanded query: {semantic_q}\n")

results_semantic = search_tfidf(semantic_q, top_k=5)
print(f"Semantic Results: {len(results_semantic)} documents\n")
for i, r in enumerate(results_semantic, 1):
    print(f"{i}. [{r['surah_id']}:{r['ayah_id']}] {r['surah_name']} (score: {r['score']:.3f})")
    print(f"   {r['text'][:80]}...\n")

---
### 2.5 — Compare All Three Methods
**What it does:** Display baseline, Rocchio, and semantic expansion results side-by-side for easy comparison

In [ ]:
def compare_methods(query: str, top_k: int = 5) -> None:
    """Compare baseline, Rocchio, and semantic expansion side-by-side."""
    print(f"Query: {query}\n")
    print("=" * 100)
    
    # Baseline
    print("\n[BASELINE TF-IDF]")
    results_base = search_tfidf(query, top_k)
    for i, r in enumerate(results_base, 1):
        print(f"{i}. [{r['surah_id']}:{r['ayah_id']}] score={r['score']:.3f}")
    
    # Rocchio
    print("\n[ROCCHIO PSEUDO RELEVANCE FEEDBACK]")
    expanded_rocchio = rocchio_expansion(query)
    results_rocchio = search_tfidf(expanded_rocchio, top_k)
    for i, r in enumerate(results_rocchio, 1):
        print(f"{i}. [{r['surah_id']}:{r['ayah_id']}] score={r['score']:.3f}")
    
    # Semantic
    print("\n[SEMANTIC EXPANSION (Sentence-Transformers)]")
    expanded_semantic = semantic_expansion(query)
    results_semantic = search_tfidf(expanded_semantic, top_k)
    for i, r in enumerate(results_semantic, 1):
        print(f"{i}. [{r['surah_id']}:{r['ayah_id']}] score={r['score']:.3f}")
    
    print("\n" + "=" * 100)

print("Comparison function defined")

**Demo:** Compare all three methods on a single query

In [ ]:
compare_methods("الله والرحمة", top_k=5)

---
### 2.6 — Unified Search API
**What it does:** 
- Single `search()` function supporting all three methods (baseline, rocchio, semantic)
- Helper `display_results()` function for consistent output formatting
- Provides consistent interface for Phase 3 (UI) development

In [ ]:
def search(query: str, method: str = 'baseline', top_k: int = 10) -> list:
    """
    Unified search API.
    
    Args:
        query (str): Arabic query text
        method (str): 'baseline' | 'rocchio' | 'semantic'
        top_k (int): number of results
    
    Returns:
        list of results with doc_id, surah_id, ayah_id, text, score
    """
    if method == 'baseline':
        return search_tfidf(query, top_k)
    elif method == 'rocchio':
        expanded_q = rocchio_expansion(query)
        return search_tfidf(expanded_q, top_k)
    elif method == 'semantic':
        expanded_q = semantic_expansion(query)
        return search_tfidf(expanded_q, top_k)
    else:
        raise ValueError(f"Unknown method: {method}")

def display_results(results: list, method_name: str = "Search") -> None:
    """Pretty-print search results."""
    print(f"\n{method_name.upper()}")
    print("─" * 90)
    for i, r in enumerate(results[:10], 1):
        print(f"{i:2d}. [{r['surah_id']:3d}:{r['ayah_id']:3d}] {r['surah_name']:<20} score={r['score']:.4f}")
        print(f"     {r['text'][:85]}")
        if i == 10:
            break
    print()

print("Unified search API defined")

**Demo:** Test unified search interface with all methods

In [ ]:
test_queries = [
    "الله",
    "النور والظلام",
    "الصلاة والعبادة"
]

print("Testing unified search interface...\n")
for q in test_queries:
    results = search(q, method='baseline', top_k=3)
    display_results(results, f"Query: {q}")

---
## Summary: Phase 1 + Phase 2

### PHASE 1: Indexing Pipeline
```
quran.json  (114 surahs, 6,236 ayahs)
  │
  ├─ Validation ──────────────────── 114 surahs, all keys present
  ├─ Corpus Builder ──────────────── 6,236 documents
  │
  ├─ Preprocessing Pipeline
  │  ├─ remove_diacritics ───────── strips tashkeel + tatweel
  │  ├─ normalize_arabic ───────── أ/إ/آ/ٱ→ا, ة→ه, ى/ئ→ي, ؤ→و
  │  ├─ tokenize ────────────────── whitespace split, drop non-Arabic
  │  ├─ remove_stopwords ───────── 100+ Arabic function words
  │  └─ stem_words (ISRI) ──────── light prefix/suffix stripping
  │
  └─ Inverted Index ──────────────── term → { df, postings }
```

### PHASE 2: Ranked Retrieval & Query Expansion
```
User Query
  │
  ├─ [Method 1] TF-IDF Baseline ─────────── PyTerrier BM25
  │
  ├─ [Method 2] Rocchio PRF ────────────── Pseudo Relevance Feedback
  │  └─ Rocchio expansion using top-k doc centroids
  │
  ├─ [Method 3] Semantic Expansion ─────── Sentence-Transformers
  │  └─ Find similar terms via multilingual embeddings
  │
  └─ Results Display ────────────────────── [surah:ayah] + text + score
```

| Component | Type | Input | Output |
|-----------|------|-------|--------|
| `preprocess()` | Function | Arabic text | (normalized_text, tokens) |
| `search_tfidf()` | Function | query string | top-k results list |
| `rocchio_expansion()` | Function | query string | expanded query |
| `semantic_expansion()` | Function | query string | expanded query |
| `search()` | API | query, method | results list |
| `compare_methods()` | Util | query | 3-method comparison |
